# Simple META API Request Test

This notebook submits a test request to the META API endpoint with the exact parameters specified.

## Setup: Import Modules

In [ ]:
import sys
from pathlib import Path
import json

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
# Import required modules
from src.scheduler.scheduler_settings import SchedulerSettings
from src.decorators import handle_api_request
from src.debug_config import DebugConfig

print("✓ Imports successful")

## Step 1: Load Configuration

Load API credentials from environment variables.

In [ ]:
# Initialize settings to get API key from environment
settings = SchedulerSettings.initialize()
ready_agents = settings.get_ready_agents()

# Get META agent configuration
meta_config = ready_agents.get("META")

if not meta_config:
    print("❌ META agent not configured")
    print(f"Available agents: {list(ready_agents.keys())}")
else:
    print(f"✓ META agent configured")
    print(f"  Endpoint: {meta_config.api_endpoint}")
    print(f"  API Key present: {bool(meta_config.api_key)}")

## Step 2: Prepare Request Parameters

Set up the exact request parameters for the META API search.

In [ ]:
# Request parameters (exact specification)
url = "https://api.metaso.cn/search"
method = "POST"
description = "meta_search"

# JSON body
json_body = {
    "q": "人工智能 大模型",
    "scope": "webpage",
    "size": "5",
    "includeSummary": False,
    "includeRawContent": True,
    "conciseSnippet": False
}

# Request headers
headers = {
    "Authorization": f"Bearer {meta_config.api_key}",
    "Accept": "application/json",
    "Content-Type": "application/json"
}

# Timeout
timeout = 30

print("✓ Request parameters prepared")
print(f"\n  URL: {url}")
print(f"  Method: {method}")
print(f"  Description: {description}")
print(f"  Timeout: {timeout}s")
print(f"\n  JSON Body:")
print(json.dumps(json_body, indent=4, ensure_ascii=False))
print(f"\n  Headers:")
print(json.dumps({
    "Authorization": "Bearer [API_KEY_HIDDEN]",
    "Accept": headers["Accept"],
    "Content-Type": headers["Content-Type"]
}, indent=4))

## Step 3: Submit Request

Call the centralized `handle_api_request()` function to submit the request.

In [ ]:
# Enable debug mode for response handling
DebugConfig.DEBUG = True

# Submit request using centralized handler
print("\n📡 Submitting request to META API...\n")

try:
    response = handle_api_request(
        agent_name="META",
        url=url,
        method=method,
        description=description,
        json_body=json_body,
        headers=headers,
        timeout=timeout,
        query_request=None
    )
    
    print("✓ Request completed successfully")
    
except Exception as e:
    print(f"❌ Request failed: {e}")
    import traceback
    traceback.print_exc()
    response = None

## Step 4: Display Response

Show the full API response.

In [ ]:
if response:
    print("\n📋 Full Response:")
    print("\n" + "="*60)
    print(json.dumps(response, indent=2, ensure_ascii=False))
    print("="*60)
else:
    print("\n⚠ No response received")

## Step 5: Analyze Response Structure

Parse and display key information from the response.

In [ ]:
if response and isinstance(response, dict):
    print("\n🔍 Response Analysis:")
    print(f"\nTop-level keys: {list(response.keys())}")
    
    # Check for webpages (META API response structure)
    if 'webpages' in response:
        webpages = response['webpages']
        print(f"\nWebpages Info:")
        print(f"  Type: {type(webpages)}")
        print(f"  Keys: {list(webpages.keys()) if isinstance(webpages, dict) else 'N/A'}")
        
        if isinstance(webpages, dict) and 'value' in webpages:
            results = webpages['value']
            print(f"\n  Number of results: {len(results)}")
            
            if results:
                print(f"\n  First result:")
                first = results[0]
                print(f"    Title: {first.get('title', 'N/A')}")
                print(f"    URL: {first.get('url', 'N/A')}")
                print(f"    Description: {first.get('description', 'N/A')[:100]}...")
    else:
        print(f"\n⚠ Expected 'webpages' key not found in response")
        print(f"   Response structure: {json.dumps(response, indent=2, ensure_ascii=False)[:500]}...")
else:
    print("\n⚠ Response is not a valid dictionary")